In [1]:
import os
from tqdm import tqdm
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors

INPUT_FILENAME = "CycPeptMPDB_Peptide_All.csv"
DATASIZE = 50
N_CONFORMERS = 20

In [2]:
import contextlib
import joblib
from tqdm import tqdm
import time


@contextlib.contextmanager
def tqdm_joblib(tqdm_object: tqdm):
    """Context manager to patch joblib to report into tqdm progress bar.

    Args:
        tqdm_object: The tqdm instance to update.
    """

    class TqdmBatchCompletionCallBack(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallBack
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()


def parallel_run(
    func, arg_list: list, n_jobs: int = -1, desc: str = "Processing", **kwargs
) -> list:
    """Executes a function in parallel with joblib and monitors progress with tqdm.

    Args:
        func: The function to be executed in parallel.
        arg_list: A list of arguments to pass to the function.
            Each element can be a single value, a tuple (positional args),
            or a dictionary (keyword args).
        n_jobs: The number of jobs to run in parallel. Default is -1 (all CPUs).
        desc: The description for the progress bar. Default is "Processing".
        **kwargs: Additional arguments passed to joblib.Parallel.

    Returns:
        A list of results from the function execution.
    """

    def wrapper(args):
        if isinstance(args, dict):
            return func(**args)
        if isinstance(args, (list, tuple)):
            return func(*args)
        return func(args)

    with tqdm_joblib(tqdm(total=len(arg_list), desc=desc)):
        return list(
            joblib.Parallel(n_jobs=n_jobs, **kwargs)(
                joblib.delayed(wrapper)(args) for args in arg_list
            )
        )

In [3]:
def get_canonical_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol)

def generate_conformers(smiles, num):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMultipleConfs(mol, numConfs=num, params=params)
    return mol
    
def get_conformer_data(mol):
    atomic_numbers = np.array([atom.GetAtomicNum() for atom in mol.GetAtoms()])
    all_positions = np.array([conf.GetPositions() for conf in mol.GetConformers()])
    return atomic_numbers, all_positions

def calc_energies(mol):
    energies = []
    for conf_id in range(mol.GetNumConformers()):
        ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol), confId=conf_id)
        energy = ff.CalcEnergy()
        energies.append(energy)
    return np.array(energies)

In [ ]:
def process_func(smiles):
    mol = generate_conformers(smiles, N_CONFORMERS)
    energies = calc_energies(mol)
    numbers, positions = get_conformer_data(mol)
    data_info = dict(numbers=numbers, positions=positions, energies=energies)
    return (mol, data_info)
    
smiles_lst = pd.read_csv(INPUT_FILENAME)["SMILES"].apply(get_canonical_smiles).unique().tolist()[:DATASIZE]
results = parallel_run(process_func, smiles_lst)

/tmp/ipykernel_23030/2791115922.py:8: DtypeWarning: Columns (0: Detection_Limit_2) have mixed types. Specify dtype option on import or set low_memory=False.
  smiles_lst = pd.read_csv(INPUT_FILENAME)["SMILES"].apply(get_canonical_smiles).unique().tolist()[:DATASIZE]
Processing:  88%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                 | 44/50 [02:18<01:08, 11.43s/it]

In [ ]:
import hashlib

def hash_smiles(smiles):
    return hashlib.sha256(smiles.encode()).hexdigest()

out_data = []
for smiles, result in zip(smiles_lst, results):
    data_info = {"id": hash_smiles(smiles), "smiles": smiles}
    data_info.update(result[1])
    out_data.append(data_info)

In [ ]:
import h5py

def save_to_h5(molecule_data_list, output_filename="dataset.h5"):
    """
    Args:
        molecule_data_list: A list of dicts, each containing:
            - "id": str (hash of SMILES)
            - "smiles": str (input SMILES)
            - "numbers": np.array (shape: [N_atoms])
            - "positions": np.array (shape: [N_conformers, N_atoms, 3])
            - "energies": np.array (shape: [N_conformers])
    """
    with h5py.File(output_filename, "w") as f:
        for data in molecule_data_list:
            grp = f.create_group(data["id"])
            grp.attrs["smiles"] = data["smiles"]
            grp.create_dataset("numbers", data=data["numbers"], compression="gzip")
            grp.create_dataset("positions", data=data["positions"], compression="gzip")
            grp.create_dataset("energies", data=data["energies"], compression="gzip")

    print(f"Successfully saved {len(molecule_data_list)} molecules to {output_filename}")

save_to_h5(out_data)

In [ ]:
def load_from_h5(filename="dataset.h5"):
    molecule_data_list = []
    
    with h5py.File(filename, "r") as f:
        for mol_id in f.keys():
            grp = f[mol_id]
            data = {
                "id": mol_id,
                "smiles": grp.attrs["smiles"],
                "numbers": grp["numbers"][:],
                "positions": grp["positions"][:],
                "energies": grp["energies"][:],
            }
            molecule_data_list.append(data)
            
    return molecule_data_list
    
loaded_data = load_from_h5("dataset.h5")

In [ ]:
import py3Dmol

def show_mol(mol, confId):
    mb = Chem.MolToMolBlock(mol, confId=confId)
    viewer = py3Dmol.view(width=400, height=400)
    viewer.addModel(mb, 'sdf')
    viewer.setStyle({'stick': {}})
    viewer.zoomTo()
    return viewer

mol = results[0][0]
for i in range(min(mol.GetNumConformers(), 3)):
    display(show_mol(mol, i))